# RBS Finance Dashboard - Colab Launcher
Execute each cell in order to start the dashboard.

In [ ]:
# CELL 1: Install packages
!pip -q install streamlit yfinance numpy pandas matplotlib seaborn scikit-learn scipy adjustText feedparser newspaper3k requests plotly openai pyngrok

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
DRIVE_BASE = Path('/content/drive/MyDrive/RBS_app')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f'Drive ready at {DRIVE_BASE}')

In [ ]:
# CELL 3: Sync latest app.py and rbs_lib.py from GitHub
import urllib.request, sys

BRANCH = 'claude/optimize-analysis-dashboard-NZUKB'
REPO   = 'RoyARVSA/RBS_claude_finance'

for fname in ['app.py', 'rbs_lib.py']:
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{fname}'
    dst = DRIVE_BASE / fname
    try:
        urllib.request.urlretrieve(url, dst)
        print(f'OK  {fname} -> {dst}')
    except Exception as e:
        print(f'WARN {fname} download failed: {e} (using existing file if any)')

if str(DRIVE_BASE) not in sys.path:
    sys.path.insert(0, str(DRIVE_BASE))
print('Sync done.')

In [ ]:
# CELL 4: Start Streamlit + pyngrok tunnel
import subprocess, socket, time, sys
from pyngrok import conf as _nc, ngrok as _ng

PORT         = 8501
NGROK_TOKEN  = '31UVEcuXcsxDf69fAhH7e4qBLFO_6wotasJhN1ZLTwwrj2NSW'  # <- your token
NGROK_REGION = 'ap'   # ap / us / eu / jp / in / au / sa
APP_PATH     = str(DRIVE_BASE / 'app.py')

# Kill old process on port
subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
time.sleep(0.5)

# Launch Streamlit
st_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', APP_PATH,
     '--server.port', str(PORT),
     '--server.address', '0.0.0.0',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Wait for ready
t0 = time.time()
while time.time() - t0 < 60:
    try:
        socket.create_connection(('127.0.0.1', PORT), 1).close()
        print('Streamlit ready')
        break
    except OSError:
        time.sleep(0.25)

# Create ngrok tunnel
_nc.get_default().auth_token = NGROK_TOKEN
_nc.get_default().region     = NGROK_REGION
for t in _ng.get_tunnels():
    _ng.disconnect(t.public_url)

pub = _ng.connect(PORT, 'http').public_url.replace('http://', 'https://')
print(f'\nPublic URL: {pub}\nApp path:   {APP_PATH}\nTo stop:    st_proc.terminate()')